# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

In [ ]:
from typing import Dict, Union, Tuple, Any, Optional, Sequence, List
import copy
import math
import random
from dataclasses import dataclass
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [52]:
%pip install pandas numpy torch --quiet

Note: you may need to restart the kernel to use updated packages.


In [53]:
%matplotlib widget

## Data From Source Package

In [54]:
%pip install -e .. --quiet # This is broken for some reason still, figuring it out

Note: you may need to restart the kernel to use updated packages.


In [55]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"

sys.path.insert(0, str(SRC))

import autonomous_fed as afed

In [56]:
solver = afed.EnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da", override_device="cpu")

# Data Check
print (solver.historical_data.head())
print (solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Architecture

### Single Hidden Layer NARX Model

For our economy transition equations we have: ${y_t=\hat{f}^y(y_{t-1},y_{t-2},\pi_t,\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^y}$ and ${\pi_t=\hat{f}^\pi(y_t,y_{t-1},y_{t-2},\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^\pi}$

In this scenario our predictor function, ${\hat{f}}$ is an ANN (Artificial Neural Network) but it can be swapped with other nonlinear functions such as a sigmoid function or wavelet network. For our ANN predictor we have ${\hat{f}^m=b_0^m+\sum_{j=1}^h v_j^mG(\omega_j^{m'}s_t^m+b_j^m), m\in\{y,\pi}\}$

The Components of the ANN are as follows:
- ${m}$: ${\{y,\pi}\}$
- ${s_t^m}$: Input state vectors at time t.
- ${w_j^m}$: Weight vector for the j-th hidden neuron.
- ${b_j^m}$: Bias term for the current neuron.
- ${G(\cdot)}$: Activation function (nonlinear transform).
- ${v_j^m}$: Weight from hidden neuron ${j}$ to the output layer
- ${b_0^m}$: Bias at the output layer
- ${h}$: Number of hidden neurons.

As seen above the Neural Network type is a NARX Model with a single hidden layer and the activation function is the hyperbolic tangent therefore we define ${G(\cdot)}$ as follows: ${G(x)=tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}}$

As done in the reference paper the ANNs are intialized with Nguyen-Widrow Initialization. In addition to this we use the Levenberg-Marquardt Algorithm for our optimizer.

## Model Buildout

in the following cells we will replicate the Bundesbank's Neural Net as close as possible by rebuilding some of the MATLAB tools in python for use with PyTorch.

Below we set the tensor dtype to float64 for pytorch because this model is being run locally on an ARM64 Macbook Pro. We can do training through the GPU by using MPS (Metal Performance Shaders) for our torch device but MPS does not support the float64 dtype only float32. As an effort to better replicate the MATLAB behavior from the reference paper we set dtype to float64 and bound our training to CPU for the torch device.

In [57]:
DTYPE = torch.float64

torch.use_deterministic_algorithms(True)

# (Optional if you ever use CUDA)
# torch.backends.cuda.matmul.allow_tf32 = False
# torch.backends.cudnn.allow_tf32 = False

#### MinMaxScaler

#### NARX Dataset

#### Nguyen-Widrow Initialization

In our reference paper it is noted that the ANNs are initialized using the Nguyen-Widrow Method. This is done because we are using ${\tanh(x)}$ for our activation function which has saturation zones near -1 and 1. In order to properly replicate the paper and ensure neurons are placed in the high-information region of the activation function,we build a Nguyen-Widrow Initializer as follows:

$$
{w_i=\beta\times\frac{w_i^0}{||w_i^0||}, b_i~U[-\beta,\beta], \beta=0.7n_{out}^{1/n_{in}}}
$$


#### NARX MLP

Below we mimic the refernce paper's Single-hidden-layer Artificial Neural Network with n-hidden nodes, ${\tanh(x)}$ activation function, and Nguyen-Widrow initialization.

#### Levenberg-Marquardt Trainer

#### Hidden Unit Search & Final Fit

#### Final Composed Runner Function

### Figure 4 Output Gap Fit: Squared Errors (Replication)

In [68]:
print(solver.historical_data)

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
...         ...       ...   ...
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25

[80 rows x 3 columns]


### Figure 5 Inflation Fit: Squared Errors (Replication)

### Table 3 Economy Fit: Mean Squared Errors

In [67]:
from IPython.display import Markdown, display

svar_y_msq = solver.squared_errors_data["se_y_svar"].mean()
svar_pi_msq = solver.squared_errors_data["se_pi_svar"].mean()
ann_y_msq = final_y["meta"]["picked_overall_mse_raw"]
ann_pi_msq = final_pi["meta"]["picked_overall_mse_raw"]

table_three_str = fr"""
| Representation | MSE Output Gap   | MSE Inflation     | MSE Total                        |
|----------------|------------------|-------------------|----------------------------------|
| SVAR           | {svar_y_msq:.3f} | {svar_pi_msq:.3f} | {(svar_pi_msq+svar_y_msq)/2:.3f} |
| ANN            | {ann_y_msq:.3f}  | {ann_pi_msq:.3f}  | {(ann_y_msq+ann_pi_msq)/2:.3f}   |
"""

display(Markdown(table_three_str))


| Representation | MSE Output Gap   | MSE Inflation     | MSE Total                        |
|----------------|------------------|-------------------|----------------------------------|
| SVAR           | 0.211 | 0.032 | 0.122 |
| ANN            | 0.792  | 0.082  | 0.437   |


### PDP Functions

#### Figure 6: Partial Dependence Surface Plot - ANN Economy, Inflation

#### Figure 7: Partial Dependence Surface Plot - ANN Economy, Output Gap